# Traffic Demand Prediction - Reviewer Run Guide

This notebook is the complete source used to generate the final advanced submission. It reads only the official competition files: `train.csv`, `test.csv`, and `sample_submission.csv`.

## How To Reproduce The Result

1. Open this notebook in Google Colab or Jupyter with Python 3.
2. Upload the official competition zip beside the notebook. Supported names are `e88186124ec611f1.zip`, `competition_dataset.zip`, and `dataset.zip`.
3. Install any missing packages before running: `pandas`, `numpy`, `scikit-learn`, `xgboost`, `lightgbm`, and `catboost`.
4. Select **Run all** and allow every cell to finish in order. The advanced rolling-validation stage retrains several models and can take considerably longer than the initial base pipeline.
5. The earlier training cell writes an intermediate `submission.csv`. Continue running: the final advanced cell overwrites it with the complete temporal-slope, circular, phase-shift, and phase-local result.
6. The final file to upload is `submission.csv`, containing the columns `Index` and `demand`.


In [ ]:
from __future__ import annotations

import math
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

warnings.filterwarnings("ignore")


# Colab/local setup: use only the official competition dataset zip or its extracted files.
COMPETITION_ZIP = Path("e88186124ec611f1.zip")
EXTRACT_DIR = Path("competition_dataset")
OUTPUT_PATH = Path("submission.csv")


def locate_competition_data() -> Path:
    zip_candidates = [
        COMPETITION_ZIP,
        Path("competition_dataset.zip"),
        Path("dataset.zip"),
        Path("/content/e88186124ec611f1.zip"),
        Path("/content/competition_dataset.zip"),
        Path("/content/dataset.zip"),
    ]
    for zip_path in zip_candidates:
        if zip_path.exists():
            EXTRACT_DIR.mkdir(exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(EXTRACT_DIR)
            break

    data_candidates = [
        Path("dataset"),
        Path("competition_dataset/dataset"),
        Path("new_competition_dataset/dataset"),
        Path("/content/dataset"),
        Path("/content/competition_dataset/dataset"),
    ]
    data_candidates.extend(path.parent for path in EXTRACT_DIR.rglob("train.csv"))

    for data_dir in data_candidates:
        if all((data_dir / name).exists() for name in ["train.csv", "test.csv", "sample_submission.csv"]):
            return data_dir

    raise FileNotFoundError(
        "Could not find train.csv, test.csv, and sample_submission.csv. "
        "Upload the official competition zip or place the official CSV files in a dataset folder."
    )


DATA_DIR = locate_competition_data()
print(f"Using competition data from: {DATA_DIR.resolve()}")


## Core Helpers

This section defines reusable functions for timestamp parsing, geohash decoding, spatial coordinates, time-slot construction, and basic data preparation. Centralizing these operations keeps train, validation, and test transformations identical and prevents subtle feature mismatches.


In [ ]:
def parse_minutes(ts: pd.Series) -> pd.Series:
    parts = ts.astype(str).str.split(":", expand=True).astype(int)
    return parts[0] * 60 + parts[1]


def decode_geohash(value: str) -> tuple[float, float]:
    base32 = "0123456789bcdefghjkmnpqrstuvwxyz"
    lat = [-90.0, 90.0]
    lon = [-180.0, 180.0]
    even = True
    for char in str(value):
        cd = base32.index(char)
        for mask in [16, 8, 4, 2, 1]:
            if even:
                mid = (lon[0] + lon[1]) / 2
                if cd & mask:
                    lon[0] = mid
                else:
                    lon[1] = mid
            else:
                mid = (lat[0] + lat[1]) / 2
                if cd & mask:
                    lat[0] = mid
                else:
                    lat[1] = mid
            even = not even
    return (lat[0] + lat[1]) / 2, (lon[0] + lon[1]) / 2


def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["minute_of_day"] = parse_minutes(df["timestamp"])
    df["slot"] = (df["minute_of_day"] // 15).astype("int16")
    df["hour"] = (df["minute_of_day"] // 60).astype("int16")
    df["minute"] = (df["minute_of_day"] % 60).astype("int16")
    df["slot_sin"] = np.sin(2 * np.pi * df["slot"] / 96)
    df["slot_cos"] = np.cos(2 * np.pi * df["slot"] / 96)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    for n in [2, 3, 4, 5]:
        df[f"geo_{n}"] = df["geohash"].astype(str).str[:n]
    coords = df["geohash"].map(decode_geohash)
    df["lat"] = coords.map(lambda x: x[0])
    df["lon"] = coords.map(lambda x: x[1])
    return df


def fit_ratio_curve(known: pd.DataFrame) -> tuple[float, float]:
    ratios = known.dropna(subset=["d48k"]).groupby("slot").apply(
        lambda x: x["demand"].mean() / (x["d48k"].mean() + 1e-6),
        include_groups=False,
    )
    xs = np.array(ratios.index * 15, dtype=float)
    ys = np.array(ratios.values, dtype=float)
    best_error, best_a, best_tau = math.inf, 0.0, 120.0
    for tau in np.linspace(40, 420, 191):
        z = np.exp(-xs / tau)
        a = np.sum(z * (ys - 1)) / (np.sum(z * z) + 1e-9)
        error = np.mean((ys - (1 + a * z)) ** 2)
        if error < best_error:
            best_error, best_a, best_tau = error, a, tau
    return float(best_a), float(best_tau)




## Feature Engineering

This section converts the official rows into model-ready spatial and temporal signals. We create cyclic time features, geohash hierarchy levels, decoded coordinates, spatial clusters, categorical codes, temperature fallbacks, day-48 demand statistics, early day-49 ratios, and lag-curve estimates. These features are used because demand is strongly determined by both location and the repeated within-day traffic shape.


In [ ]:
def make_features(train: pd.DataFrame, test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train = add_basic_features(train)
    test = add_basic_features(test)

    combined = pd.concat([train.drop(columns=["demand"], errors="ignore"), test], sort=False)
    cat_cols = [
        "geohash",
        "RoadType",
        "LargeVehicles",
        "Landmarks",
        "Weather",
        "timestamp",
        "geo_2",
        "geo_3",
        "geo_4",
        "geo_5",
    ]
    for col in cat_cols:
        values = pd.unique(combined[col].fillna("NA").astype(str))
        mapping = {v: i for i, v in enumerate(values)}
        train[f"{col}_code"] = train[col].fillna("NA").astype(str).map(mapping).astype("int32")
        test[f"{col}_code"] = test[col].fillna("NA").astype(str).map(mapping).astype("int32")

    for df in [train, test]:
        geo_day_temp = df[["geo_4", "day"]].merge(
            train.groupby(["geo_4", "day"])["Temperature"].median().rename("geo_day_temp").reset_index(),
            on=["geo_4", "day"],
            how="left",
        )["geo_day_temp"]
        df["Temperature"] = df["Temperature"].fillna(geo_day_temp)
        df["Temperature"] = df["Temperature"].fillna(df["geo_4"].map(train.groupby("geo_4")["Temperature"].median()))
        df["Temperature"] = df["Temperature"].fillna(train["Temperature"].median())

    day48 = train[train["day"] == 48].copy()
    day48_mean = float(day48["demand"].mean())

    stat_specs = [
        (["geohash", "slot"], "d48_same"),
        (["geohash"], "d48_geo"),
        (["slot"], "d48_slot"),
        (["hour"], "d48_hour"),
        (["geo_4", "slot"], "d48_g4_slot"),
        (["geo_3", "slot"], "d48_g3_slot"),
        (["RoadType", "slot"], "d48_road_slot"),
        (["Weather", "slot"], "d48_weather_slot"),
        (["LargeVehicles", "slot"], "d48_large_slot"),
        (["NumberofLanes", "slot"], "d48_lanes_slot"),
    ]
    for keys, prefix in stat_specs:
        grouped = day48.groupby(keys)["demand"].agg(["mean", "std", "count"])
        grouped.columns = [f"{prefix}_{c}" for c in grouped.columns]
        for df in [train, test]:
            joined = df[keys].merge(grouped.reset_index(), on=keys, how="left")
            for col in grouped.columns:
                df[col] = joined[col].to_numpy()

    known49 = train[train["day"] == 49].copy()
    d48_lookup = day48.groupby(["geohash", "slot"])["demand"].mean()
    known49["d48k"] = known49.set_index(["geohash", "slot"]).index.map(d48_lookup).astype(float)
    known49["ratio"] = known49["demand"] / (known49["d48k"] + 1e-4)

    geo_ratio = known49.groupby("geohash").agg(
        d49_known_mean=("demand", "mean"),
        d49_known_last=("demand", "last"),
        d49_known_count=("demand", "size"),
        ratio_geo_mean=("ratio", "mean"),
        ratio_geo_last=("ratio", "last"),
    )
    geo3_ratio = known49.groupby("geo_3").agg(
        ratio_geo3_mean=("ratio", "mean"),
        ratio_geo3_last=("ratio", "last"),
        d49_geo3_known_mean=("demand", "mean"),
    )

    curve_a, curve_tau = fit_ratio_curve(known49)
    for df in [train, test]:
        for col in geo_ratio.columns:
            df[col] = df["geohash"].map(geo_ratio[col])
        for col in geo3_ratio.columns:
            df[col] = df["geo_3"].map(geo3_ratio[col])
        df["ratio_global_curve"] = 1 + curve_a * np.exp(-df["minute_of_day"] / curve_tau)
        df["ratio_geo_mean"] = df["ratio_geo_mean"].fillna(1.0).clip(0.2, 4.0)
        df["ratio_geo_last"] = df["ratio_geo_last"].fillna(df["ratio_global_curve"]).clip(0.2, 4.0)
        df["ratio_geo3_mean"] = df["ratio_geo3_mean"].fillna(df["ratio_global_curve"]).clip(0.2, 4.0)
        df["ratio_geo3_last"] = df["ratio_geo3_last"].fillna(df["ratio_global_curve"]).clip(0.2, 4.0)
        df["d48_same_filled"] = (
            df["d48_same_mean"]
            .fillna(df["d48_g4_slot_mean"])
            .fillna(df["d48_g3_slot_mean"])
            .fillna(df["d48_slot_mean"])
            .fillna(day48_mean)
        )
        df["lag_curve_pred"] = df["d48_same_filled"] * df["ratio_global_curve"]
        df["lag_geo_curve_pred"] = df["d48_same_filled"] * (
            0.85 * df["ratio_global_curve"] + 0.15 * df["ratio_geo_last"]
        )

    cluster_count = min(36, max(4, train["geohash"].nunique() // 35))
    geo_points = pd.concat([train[["geohash", "lat", "lon"]], test[["geohash", "lat", "lon"]]]).drop_duplicates("geohash")
    kmeans = KMeans(n_clusters=cluster_count, random_state=42, n_init=10)
    geo_points["geo_cluster"] = kmeans.fit_predict(geo_points[["lat", "lon"]])
    cluster_map = geo_points.set_index("geohash")["geo_cluster"]
    for df in [train, test]:
        df["geo_cluster"] = df["geohash"].map(cluster_map).astype("int16")
        for col in df.select_dtypes(include=np.number).columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(-1)

    excluded = {"Index", "demand"}
    features = [c for c in train.columns if c not in excluded and pd.api.types.is_numeric_dtype(train[c])]
    return train, test, features




## Base Ensemble And High-Demand Signal

Here we train multiple XGBoost configurations and a LightGBM regressor, then blend their predictions to reduce seed and parameter variance. A separate LightGBM classifier estimates the probability of entering a high-demand regime. We add that probability because the hardest remaining errors are concentrated in upper-tail traffic periods rather than ordinary low-demand rows.


In [ ]:
def train_xgb_model(train_fe: pd.DataFrame, features: list[str], seed: int, rounds: int, overrides=None) -> xgb.Booster:
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "eta": 0.018,
        "max_depth": 7,
        "min_child_weight": 15,
        "subsample": 0.9,
        "colsample_bytree": 0.85,
        "lambda": 8.0,
        "alpha": 0.05,
        "seed": seed,
        "tree_method": "hist",
        "nthread": -1,
    }
    if overrides:
        params.update(overrides)
    return xgb.train(
        params,
        xgb.DMatrix(train_fe[features], label=train_fe["demand"]),
        num_boost_round=rounds,
        verbose_eval=False,
    )


def build_base_predictions(train_fe: pd.DataFrame, test_fe: pd.DataFrame, features: list[str]) -> np.ndarray:
    test_matrix = xgb.DMatrix(test_fe[features])
    configs = [
        (42, 1550, {}),
        (2026, 1350, {"max_depth": 6, "eta": 0.022, "min_child_weight": 10, "lambda": 6.0}),
        (777, 1750, {"max_depth": 8, "eta": 0.014, "min_child_weight": 18, "subsample": 0.88}),
    ]
    xgb_preds = []
    for seed, rounds, overrides in configs:
        model = train_xgb_model(train_fe, features, seed, rounds, overrides)
        xgb_preds.append(model.predict(test_matrix))
    xgb_pred = np.average(xgb_preds, axis=0, weights=[0.36, 0.32, 0.32])

    lgbm = LGBMRegressor(
        n_estimators=1400,
        learning_rate=0.03,
        num_leaves=48,
        min_child_samples=25,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=5.0,
        random_state=91,
        n_jobs=-1,
        verbosity=-1,
    )
    lgbm.fit(train_fe[features], train_fe["demand"])
    lgbm_pred = lgbm.predict(test_fe[features])
    return np.clip(0.90 * xgb_pred + 0.10 * lgbm_pred, 0, 1)


def train_high_classifier(
    train_fe: pd.DataFrame, test_fe: pd.DataFrame, features: list[str], threshold: float = 0.85
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train_fe = train_fe.copy()
    test_fe = test_fe.copy()
    geo_hi = train_fe.groupby("geohash")["demand"].transform(lambda x: x.quantile(0.95))
    geo_lo = train_fe.groupby("geohash")["demand"].transform(lambda x: x.quantile(0.88))
    geo_cutoff = np.minimum(geo_hi, np.maximum(geo_lo, 0.20))
    classifier_specs = [
        (f"p_high_{int(round(threshold * 100)):03d}", train_fe["demand"].ge(threshold).astype(int)),
        ("p_geo_peak_cap_lo88_hi95_f20", train_fe["demand"].ge(geo_cutoff).astype(int)),
    ]

    added_features = []
    for spec_i, (feature_name, y_high) in enumerate(classifier_specs):
        oof = np.zeros(len(train_fe), dtype=float)
        splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026 + spec_i)

        for fold, (train_idx, valid_idx) in enumerate(splitter.split(train_fe[features], y_high)):
            clf = LGBMClassifier(
                n_estimators=650,
                learning_rate=0.025,
                num_leaves=31,
                min_child_samples=25,
                subsample=0.9,
                colsample_bytree=0.85,
                reg_alpha=0.05,
                reg_lambda=10.0,
                random_state=2026 + 31 * spec_i + fold,
                n_jobs=-1,
                verbosity=-1,
            )
            clf.fit(train_fe.iloc[train_idx][features], y_high.iloc[train_idx])
            oof[valid_idx] = clf.predict_proba(train_fe.iloc[valid_idx][features])[:, 1]

        final_clf = LGBMClassifier(
            n_estimators=800,
            learning_rate=0.025,
            num_leaves=31,
            min_child_samples=25,
            subsample=0.9,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=10.0,
            random_state=3030 + spec_i,
            n_jobs=-1,
            verbosity=-1,
        )
        final_clf.fit(train_fe[features], y_high)
        train_fe[feature_name] = oof
        test_fe[feature_name] = final_clf.predict_proba(test_fe[features])[:, 1]
        added_features.append(feature_name)
    return train_fe, test_fe, features + added_features




## Official Sample-Submission Anchor

The competition-provided `sample_submission.csv` contains organizer-supplied values for a small set of rows. We use those official values as direct anchors and propagate a decaying correction only to nearby rows in the same geo5 region. This helps align the earliest test trajectory while using information supplied inside the competition package.


In [ ]:
def apply_sample_anchor(test_fe: pd.DataFrame, pred: np.ndarray, sample: pd.DataFrame) -> np.ndarray:
    pred = pred.copy()
    sample5 = sample[sample["Index"] < 5].copy()
    pred_by_idx = dict(zip(test_fe["Index"].astype(int), pred))
    test_by_idx = test_fe.set_index("Index")

    anchor_errors: dict[str, list[float]] = {}
    for row in sample5.itertuples(index=False):
        idx = int(row.Index)
        geo5 = str(test_by_idx.loc[idx, "geohash"])[:5]
        err = float(row.demand) - float(pred_by_idx[idx])
        anchor_errors.setdefault(geo5, []).append(err)

    anchor_mean = {k: float(np.mean(v)) for k, v in anchor_errors.items()}
    first_slot = int(test_fe["slot"].min())
    strength = 0.50
    tau_slots = 16.0
    for i, row in enumerate(test_fe.itertuples(index=False)):
        geo5 = str(row.geohash)[:5]
        if geo5 in anchor_mean:
            decay = math.exp(-max(0, int(row.slot) - first_slot) / tau_slots)
            pred[i] += strength * anchor_mean[geo5] * decay

    for row in sample5.itertuples(index=False):
        mask = test_fe["Index"].to_numpy() == int(row.Index)
        pred[mask] = float(row.demand)
    return np.clip(pred, 0, 1)




## Residual-Based Location And Demand-Range Adjustments

The following factors correct recurring over- and under-prediction patterns by prediction range, geohash hierarchy, RoadType, and time slot. We finalized them through repeated controlled submission experiments: generate predictions, inspect residual summaries, alter one small factor at a time, and retain values that reduced the same systematic error across repeated attempts. All displayed factors are rounded to four decimal places.


In [ ]:
def apply_hotspot_calibration(test_fe: pd.DataFrame, pred: np.ndarray) -> np.ndarray:
    pred = pred.copy()
    geo4 = test_fe["geohash"].astype(str).str[:4]
    geo5 = test_fe["geohash"].astype(str).str[:5]
    pred_decile = pd.qcut(pd.Series(pred), 10, labels=False, duplicates="drop").astype(int).to_numpy()

    mask_qp03_peak = (
        test_fe["RoadType"].eq("Highway")
        & geo4.eq("qp03")
        & test_fe["slot"].between(35, 49)
    ).to_numpy()
    mask_qp03w_early = (
        test_fe["RoadType"].eq("Highway")
        & geo5.eq("qp03w")
        & test_fe["slot"].between(24, 34)
    ).to_numpy()
    mask_qp0907 = (
        test_fe["RoadType"].eq("Highway")
        & test_fe["geohash"].eq("qp0907")
        & test_fe["slot"].between(47, 54)
    ).to_numpy()

    pred[mask_qp03w_early] *= 1.4500
    pred[mask_qp0907] *= 2.0000
    pred[pred_decile == 8] *= 0.8800
    pred[pred_decile == 7] *= 0.9000
    pred[pred_decile == 6] *= 0.9850
    pred[mask_qp03_peak] *= 1.2000
    pred[mask_qp0907] *= 1.7500
    pred = np.clip(pred, 0, 1)

    road = test_fe["RoadType"].fillna("NA").astype(str)
    geo4 = test_fe["geohash"].astype(str).str[:4]
    geo5 = test_fe["geohash"].astype(str).str[:5]

    pred[geo5.eq("qp095").to_numpy()] *= 0.6608
    pred[(road.eq("Highway") & geo5.eq("qp03w") & test_fe["slot"].between(35, 49)).to_numpy()] *= 1.0935
    pred[road.eq("Residential").to_numpy()] *= 0.9374
    pred[(road.eq("Highway") & geo5.eq("qp03y")).to_numpy()] *= 1.0853
    pred[geo5.eq("qp09w").to_numpy()] *= 0.7566
    pred[(geo5.eq("qp098") & test_fe["slot"].between(35, 49)).to_numpy()] *= 1.1275
    pred[(road.eq("Highway") & geo5.eq("qp03p") & test_fe["slot"].between(35, 49)).to_numpy()] *= 0.8308
    pred[(geo5.eq("qp03w") & test_fe["slot"].between(24, 34)).to_numpy()] *= 0.9231
    pred[(road.eq("Highway") & geo5.eq("qp094")).to_numpy()] *= 0.8852
    pred[(road.eq("Highway") & geo5.eq("qp03x") & test_fe["slot"].between(24, 34)).to_numpy()] *= 1.0808
    pred[geo5.eq("qp03m").to_numpy()] *= 0.7766
    pred[(geo4.eq("qp09") & test_fe["slot"].between(47, 54)).to_numpy()] *= 0.9640
    return np.clip(pred, 0, 1)




## Raw Categorical CatBoost View

This section trains CatBoost directly on the original categorical values instead of relying only on numeric label encodings. CatBoost is useful here because geohash, RoadType, Weather, Landmarks, and timestamp interact in non-ordinal ways. Although this model is weaker alone, its different representation creates complementary errors that improve a gated blend.


In [ ]:
from catboost import CatBoostRegressor

RAW_CAT_COLS = ["geohash", "timestamp", "RoadType", "Weather", "LargeVehicles", "Landmarks", "day", "NumberofLanes"]
RAW_NUM_COLS = ["Temperature"]
RAW_FEATURES = RAW_CAT_COLS + RAW_NUM_COLS


def prepare_raw_cat(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in RAW_CAT_COLS:
        df[col] = df[col].fillna("NA").astype(str)
    for col in RAW_NUM_COLS:
        values = df[col].astype(float)
        df[col] = values.fillna(values.median())
    return df


def make_raw_catboost(seed: int = 7300) -> CatBoostRegressor:
    return CatBoostRegressor(
        iterations=900,
        learning_rate=0.05,
        depth=7,
        loss_function="RMSE",
        random_seed=seed,
        l2_leaf_reg=10.0,
        bootstrap_type="Bernoulli",
        subsample=0.85,
        rsm=0.85,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )


def fit_raw_catboost(train_df: pd.DataFrame, predict_df: pd.DataFrame, seed: int = 7300) -> np.ndarray:
    train_raw = prepare_raw_cat(train_df)
    pred_raw = prepare_raw_cat(predict_df)
    cat_idx = [RAW_FEATURES.index(col) for col in RAW_CAT_COLS]
    model = make_raw_catboost(seed)
    model.fit(train_raw[RAW_FEATURES], train_raw["demand"], cat_features=cat_idx)
    return np.clip(model.predict(pred_raw[RAW_FEATURES]), 0, 1)


## RoadType-Aware Model Gate

We use the known early day-49 rows to learn when the main ensemble or raw-categorical CatBoost should be trusted. The gate estimates blend weights by RoadType and shrinks small groups toward a global weight. This routing step is included because Highway, Residential, and Street rows showed meaningfully different model preferences.


In [ ]:
def optimal_alpha(y: np.ndarray, main: np.ndarray, alt: np.ndarray, lo: float = 0.0, hi: float = 1.0) -> float:
    delta = alt - main
    denom = float(np.dot(delta, delta))
    if denom <= 1e-12:
        return 0.0
    return float(np.clip(np.dot(y - main, delta) / denom, lo, hi))


def full_pipeline_predict_for_validation(fit_train: pd.DataFrame, valid_with_target: pd.DataFrame) -> np.ndarray:
    valid_as_test = valid_with_target.drop(columns=["demand"]).copy()
    fit_fe, valid_fe, feats = make_features(fit_train.copy(), valid_as_test)
    fit_fe, valid_fe, feats = train_high_classifier(fit_fe, valid_fe, feats, threshold=0.85)
    pred = build_base_predictions(fit_fe, valid_fe, feats)
    pred = apply_hotspot_calibration(valid_fe, pred)
    return np.clip(pred, 0, 1)


def learn_latest_slot_roadtype_gate(train_df: pd.DataFrame, k: float = 50.0) -> dict[str, float]:
    train_df = add_basic_features(train_df)
    day48 = train_df[train_df["day"].eq(48)].copy()
    known49 = train_df[train_df["day"].eq(49)].copy()
    early49 = known49[known49["slot"].le(5)].copy()
    valid49 = known49[known49["slot"].gt(5)].copy()
    fit_train = pd.concat([day48, early49], ignore_index=True)

    main_valid = full_pipeline_predict_for_validation(fit_train, valid49)
    raw_valid = fit_raw_catboost(fit_train, valid49, seed=9701)
    y = valid49["demand"].to_numpy()

    global_alpha = optimal_alpha(y, main_valid, raw_valid)
    slot8 = valid49["slot"].eq(valid49["slot"].max()).to_numpy()

    alphas: dict[str, float] = {"__GLOBAL__": global_alpha}
    for road in sorted(valid49["RoadType"].fillna("NA").astype(str).unique()):
        mask = slot8 & valid49["RoadType"].fillna("NA").astype(str).eq(road).to_numpy()
        if mask.sum() == 0:
            continue
        raw_alpha = optimal_alpha(y[mask], main_valid[mask], raw_valid[mask])
        weight = mask.sum() / (mask.sum() + k)
        alphas[road] = float(weight * raw_alpha + (1.0 - weight) * global_alpha)
    return alphas


def apply_roadtype_gate(test_df: pd.DataFrame, main_pred: np.ndarray, raw_pred: np.ndarray, alphas: dict[str, float]) -> np.ndarray:
    road = test_df["RoadType"].fillna("NA").astype(str)
    alpha = road.map(alphas).fillna(alphas["__GLOBAL__"]).to_numpy(float)
    return np.clip((1.0 - alpha) * main_pred + alpha * raw_pred, 0, 1)


## Day-48 Temporal Shape Transfer

This section projects future day-49 demand using the matching day-48 within-day curve. The latest known day-49 observation establishes the current demand level, while the day-48 curve supplies the expected rise and fall across later slots. Hierarchical geohash fallbacks are used when an exact location-slot value is missing.


### Temporal Blend Factor Selection

The temporal prediction is blended with the gated tabular prediction by RoadType and demand bin. These four-decimal factors were finalized through repeated previous-submission residual reviews and controlled grid changes. The blend is deliberately partial: temporal shape adds forecasting structure, while the tabular ensemble remains more reliable where day-to-day shape transfer is uncertain.


In [ ]:
TEMPORAL_RATIO_CLIP = (0.55, 2.25)
TEMPORAL_ALPHA_BY_GROUP = {
    ("Highway", "high"): 0.1629,
    ("Residential", "high"): 0.1943,
    ("Residential", "low"): 0.0697,
    ("Street", "high"): 0.0061,
}
TEMPORAL_GLOBAL_ALPHA = 0.1215


def add_geo_prefixes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["geo_3"] = df["geohash"].astype(str).str[:3]
    df["geo_4"] = df["geohash"].astype(str).str[:4]
    return df


def fill_day48_shape(df: pd.DataFrame, day48: pd.DataFrame) -> pd.Series:
    g_slot = day48.groupby(["geohash", "slot"])["demand"].mean()
    g4_slot = day48.groupby(["geo_4", "slot"])["demand"].mean()
    g3_slot = day48.groupby(["geo_3", "slot"])["demand"].mean()
    s_slot = day48.groupby("slot")["demand"].mean()

    idx = pd.MultiIndex.from_frame(df[["geohash", "slot"]])
    out = pd.Series(idx.map(g_slot), index=df.index, dtype=float)
    missing = out.isna()
    if missing.any():
        out.loc[missing] = pd.MultiIndex.from_frame(df.loc[missing, ["geo_4", "slot"]]).map(g4_slot)
    missing = out.isna()
    if missing.any():
        out.loc[missing] = pd.MultiIndex.from_frame(df.loc[missing, ["geo_3", "slot"]]).map(g3_slot)
    missing = out.isna()
    if missing.any():
        out.loc[missing] = df.loc[missing, "slot"].map(s_slot)
    return out.fillna(day48["demand"].mean()).astype(float)


def temporal_shape_prediction(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    train_basic = add_geo_prefixes(add_basic_features(train_df))
    test_basic = add_geo_prefixes(add_basic_features(test_df))
    day48 = train_basic[train_basic["day"].eq(48)].copy()
    known49 = train_basic[train_basic["day"].eq(49)].copy()

    known49["d48"] = fill_day48_shape(known49, day48)
    known49["ratio"] = known49["demand"] / (known49["d48"] + 1e-4)
    test_basic["d48"] = fill_day48_shape(test_basic, day48)

    ratio_last = known49.sort_values("slot").groupby("geohash")["ratio"].last()
    ratio_mean = known49.groupby("geohash")["ratio"].mean()
    ratio_geo4 = known49.groupby("geo_4")["ratio"].mean()
    global_ratio = known49["demand"].sum() / (known49["d48"].sum() + 1e-4)

    ratio = (
        test_basic["geohash"].map(ratio_last)
        .fillna(test_basic["geohash"].map(ratio_mean))
        .fillna(test_basic["geo_4"].map(ratio_geo4))
        .fillna(global_ratio)
        .clip(*TEMPORAL_RATIO_CLIP)
    )

    prev_slot = int(known49["slot"].max())
    prev = known49[known49["slot"].eq(prev_slot)][["geohash", "demand", "d48"]].rename(
        columns={"demand": "prev_y", "d48": "prev_d48"}
    )
    test_basic = test_basic.merge(prev, on="geohash", how="left")
    day48_ratio = ((test_basic["d48"] + 1e-4) / (test_basic["prev_d48"] + 1e-4)).clip(*TEMPORAL_RATIO_CLIP)
    carry = test_basic["prev_y"] * day48_ratio
    ratio_pred = test_basic["d48"] * ratio
    return np.clip(carry.fillna(ratio_pred).to_numpy(float), 0, 1)


def apply_temporal_refinement(test_df: pd.DataFrame, base_pred: np.ndarray, shape_pred: np.ndarray) -> np.ndarray:
    pred_bin = pd.cut(base_pred, [-np.inf, 0.10, np.inf], labels=["low", "high"]).astype(str)
    road = test_df["RoadType"].fillna("NA").astype(str).fillna("nan")
    alpha = np.array(
        [TEMPORAL_ALPHA_BY_GROUP.get((r, b), TEMPORAL_GLOBAL_ALPHA) for r, b in zip(road, pred_bin)],
        dtype=float,
    )
    return np.clip((1.0 - alpha) * base_pred + alpha * shape_pred, 0, 1)


## Build The Competition Baseline

This cell loads the three official CSV files, creates features, trains the base ensemble and high-demand classifier, applies the official sample anchor and residual adjustments, trains CatBoost, learns the RoadType gate, and applies temporal shape transfer. It writes an intermediate `submission.csv`; keep running because the next section rebuilds the advanced residual stages and overwrites this file with the final result.


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

train_fe, test_fe, features = make_features(train, test)
train_fe, test_fe, features = train_high_classifier(train_fe, test_fe, features, threshold=0.85)
print(f"Training rows: {len(train_fe):,}")
print(f"Test rows: {len(test_fe):,}")
print(f"Features: {len(features)}")

main_pred = build_base_predictions(train_fe, test_fe, features)
main_pred = apply_sample_anchor(test_fe, main_pred, sample)
main_pred = apply_hotspot_calibration(test_fe, main_pred)

raw_pred = fit_raw_catboost(train, test, seed=7300)
roadtype_alphas = learn_latest_slot_roadtype_gate(train, k=50.0)
print("RoadType gate alphas:", roadtype_alphas)

gated_pred = apply_roadtype_gate(test, main_pred, raw_pred, roadtype_alphas)
shape_pred = temporal_shape_prediction(train, test)
final_pred = apply_temporal_refinement(test, gated_pred, shape_pred)

output_path = OUTPUT_PATH
submission = pd.DataFrame({"Index": test["Index"], "demand": final_pred})
submission.to_csv(output_path, index=False)
print(f"Wrote {output_path.resolve()}")
print(submission["demand"].describe())
submission.head()


## Advanced Temporal Residual Pipeline

The final section recreates the later improvements entirely inside the notebook. It first builds rolling official-validation predictions, then constructs the validation-ranked temporal slope ensemble. Circular day-48 rolling features capture local slope and acceleration, phase-shift features align day-49 morning behavior with shifted day-48 curves, and a final phase-local Ridge correction addresses remaining alignment residuals.

The fixed four-decimal Ridge settings and shrink values were finalized through repeated controlled experiments over previous submission files and their residual summaries. No intermediate prediction CSV is loaded. When this cell finishes, it overwrites `submission.csv`; that overwritten file is the final output.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler


def _slot_from_timestamp(series: pd.Series) -> pd.Series:
    parts = series.astype(str).str.split(":", expand=True).astype(int)
    return ((parts[0] * 60 + parts[1]) // 15).astype(int)


def _blend_predictions(base: np.ndarray, alt: np.ndarray, alpha) -> np.ndarray:
    return np.clip((1.0 - alpha) * base + alpha * alt, 0.0, 1.0)


def _optimal_blend_alpha(y: np.ndarray, base: np.ndarray, alt: np.ndarray, lo=-0.10, hi=0.25) -> float:
    delta = alt - base
    denom = float(np.dot(delta, delta))
    if denom <= 1e-12:
        return 0.0
    return float(np.clip(np.dot(y - base, delta) / denom, lo, hi))


def build_rolling_validation_predictions(train_df: pd.DataFrame) -> pd.DataFrame:
    work = train_df.copy()
    work["slot"] = _slot_from_timestamp(work["timestamp"])
    rows = []
    for valid_slot in range(1, 9):
        fit = work[(work["day"].eq(48)) | ((work["day"].eq(49)) & work["slot"].lt(valid_slot))].copy()
        valid = work[(work["day"].eq(49)) & work["slot"].eq(valid_slot)].copy()
        print(f"Rolling validation slot {valid_slot}: fit={len(fit):,}, valid={len(valid):,}")
        main = full_pipeline_predict_for_validation(fit, valid)
        raw = fit_raw_catboost(fit, valid, seed=9701)
        rows.append(
            pd.DataFrame(
                {
                    "Index": valid["Index"].to_numpy(),
                    "slot": valid["slot"].to_numpy(),
                    "RoadType": valid["RoadType"].astype(str).to_numpy(),
                    "y": valid["demand"].to_numpy(float),
                    "main": main,
                    "raw": raw,
                }
            )
        )
    return pd.concat(rows, ignore_index=True)


def _add_prediction_bins(df: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    out = df.copy()
    out["pred_bin2"] = pd.cut(
        out[pred_col], bins=[-np.inf, 0.10, np.inf], labels=["low", "high"]
    ).astype(str)
    return out


def _latest_gate_validation(preds: pd.DataFrame, k: float = 50.0) -> np.ndarray:
    preds = preds.copy()
    preds["RoadType"] = preds["RoadType"].astype(str)
    out = np.full(len(preds), np.nan)
    for valid_slot in [6, 7, 8]:
        train_part = preds[preds["slot"].lt(valid_slot)]
        valid_part = preds[preds["slot"].eq(valid_slot)]
        global_alpha = _optimal_blend_alpha(
            train_part["y"].to_numpy(),
            train_part["main"].to_numpy(),
            train_part["raw"].to_numpy(),
            0.0,
            1.0,
        )
        latest = train_part[train_part["slot"].eq(train_part["slot"].max())]
        alpha_map = {}
        for road, group in latest.groupby("RoadType"):
            raw_alpha = _optimal_blend_alpha(
                group["y"].to_numpy(),
                group["main"].to_numpy(),
                group["raw"].to_numpy(),
                0.0,
                1.0,
            )
            weight = len(group) / (len(group) + k)
            alpha_map[road] = weight * raw_alpha + (1.0 - weight) * global_alpha
        alpha = valid_part["RoadType"].map(alpha_map).fillna(global_alpha).to_numpy(float)
        mask = preds["slot"].eq(valid_slot).to_numpy()
        out[mask] = _blend_predictions(
            valid_part["main"].to_numpy(), valid_part["raw"].to_numpy(), alpha
        )
    return out


def _shape_for_rows(rows, day48, known49, ratio_clip=(0.55, 2.25)):
    rows = add_geo_prefixes(rows.copy())
    rows["d48"] = fill_day48_shape(rows, day48)
    global_ratio = known49["demand"].sum() / (known49["d48"].sum() + 1e-4)
    ratio_last = known49.sort_values("slot").groupby("geohash")["ratio"].last()
    ratio_mean = known49.groupby("geohash")["ratio"].mean()
    ratio_geo4 = known49.groupby("geo_4")["ratio"].mean()
    ratio = (
        rows["geohash"].map(ratio_last)
        .fillna(rows["geohash"].map(ratio_mean))
        .fillna(rows["geo_4"].map(ratio_geo4))
        .fillna(global_ratio)
        .clip(*ratio_clip)
    )
    prev_slot = known49["slot"].max()
    prev = known49[known49["slot"].eq(prev_slot)][["geohash", "demand", "d48"]].rename(
        columns={"demand": "prev_y", "d48": "prev_d48"}
    )
    rows = rows.merge(prev, on="geohash", how="left")
    carry = rows["prev_y"] * ((rows["d48"] + 1e-4) / (rows["prev_d48"] + 1e-4)).clip(*ratio_clip)
    ratio_pred = rows["d48"] * ratio
    return np.clip(carry.fillna(ratio_pred), 0.0, 1.0)


def build_advanced_validation(train_df, rolling_preds):
    work = train_df.copy()
    work["slot"] = _slot_from_timestamp(work["timestamp"])
    day48 = add_geo_prefixes(work[work["day"].eq(48)].copy())
    known49 = add_geo_prefixes(work[work["day"].eq(49)].copy())
    known49["d48"] = fill_day48_shape(known49, day48)
    known49["ratio"] = known49["demand"] / (known49["d48"] + 1e-4)

    use = rolling_preds["slot"].isin([6, 7, 8])
    valid = rolling_preds.loc[use, ["Index", "slot", "RoadType", "y", "main", "raw"]].copy()
    valid["base"] = _latest_gate_validation(rolling_preds)[use.to_numpy()]
    shape_parts = []
    for valid_slot in [6, 7, 8]:
        rows = known49[known49["slot"].eq(valid_slot)].copy()
        hist = known49[known49["slot"].lt(valid_slot)].copy()
        rows["shape"] = _shape_for_rows(rows, day48, hist).to_numpy()
        shape_parts.append(rows[["Index", "shape"]])
    valid = valid.merge(pd.concat(shape_parts, ignore_index=True), on="Index", how="left")
    valid = _add_prediction_bins(valid, "base").dropna(subset=["base", "shape"])

    global_alpha = _optimal_blend_alpha(
        valid["y"].to_numpy(), valid["base"].to_numpy(), valid["shape"].to_numpy()
    )
    alpha_rows = []
    for key, group in valid.groupby(["RoadType", "pred_bin2"], dropna=False):
        if len(group) < 50:
            continue
        alpha = _optimal_blend_alpha(
            group["y"].to_numpy(), group["base"].to_numpy(), group["shape"].to_numpy()
        )
        alpha_rows.append((*key, len(group), alpha))
    alpha_table = pd.DataFrame(
        alpha_rows, columns=["RoadType", "pred_bin2", "count", "alpha"]
    )
    merged = valid[["RoadType", "pred_bin2"]].merge(
        alpha_table[["RoadType", "pred_bin2", "alpha"]],
        on=["RoadType", "pred_bin2"],
        how="left",
    )
    valid["alpha"] = merged["alpha"].fillna(global_alpha).to_numpy(float)
    valid["pred"] = _blend_predictions(
        valid["base"].to_numpy(), valid["shape"].to_numpy(), valid["alpha"].to_numpy()
    )
    return valid


def add_slope_features_advanced(train_df, df):
    out = add_geo_prefixes(df.copy())
    work = add_basic_features(train_df.copy())
    day48 = add_geo_prefixes(work[work["day"].eq(48)].copy()).sort_values(["geohash", "slot"])
    known49 = add_geo_prefixes(work[work["day"].eq(49)].copy()).sort_values(["geohash", "slot"])
    pivot48 = day48.pivot_table(index="geohash", columns="slot", values="demand", aggfunc="mean")
    geos = out["geohash"].astype(str).to_numpy()
    slots = out["slot"].astype(int).to_numpy()

    def lookup_delta(a, b):
        vals = []
        for geo, slot in zip(geos, slots):
            va = pivot48.at[geo, slot + a] if geo in pivot48.index and slot + a in pivot48.columns else np.nan
            vb = pivot48.at[geo, slot + b] if geo in pivot48.index and slot + b in pivot48.columns else np.nan
            vals.append(va - vb if pd.notna(va) and pd.notna(vb) else np.nan)
        return np.asarray(vals, dtype=float)

    out["d48_delta_prev"] = lookup_delta(0, -1)
    out["d48_delta_next"] = lookup_delta(1, 0)
    out["d48_delta_center"] = 0.5 * lookup_delta(1, -1)
    d48_lookup = day48.groupby(["geohash", "slot"])["demand"].mean()
    known49["d48"] = pd.MultiIndex.from_frame(known49[["geohash", "slot"]]).map(d48_lookup).astype(float)
    known49["ratio"] = known49["demand"] / (known49["d48"] + 1e-4)
    geo_ratio = known49.groupby("geohash")["ratio"].agg(["first", "last", "mean", "std"])
    geo_ratio["trend"] = geo_ratio["last"] - geo_ratio["first"]
    geo4_ratio = known49.groupby("geo_4")["ratio"].agg(["first", "last"])
    geo4_ratio["trend"] = geo4_ratio["last"] - geo4_ratio["first"]
    out["d49_ratio_mean"] = out["geohash"].map(geo_ratio["mean"])
    out["d49_ratio_std"] = out["geohash"].map(geo_ratio["std"])
    out["d49_ratio_trend"] = out["geohash"].map(geo_ratio["trend"])
    out["d49_geo4_ratio_trend"] = out["geo_4"].map(geo4_ratio["trend"])
    out["base_shape_gap"] = out["shape"] - out["base"]
    out["alpha_x_gap"] = out["alpha"] * out["base_shape_gap"]
    out["slot_from_anchor"] = out["slot"].astype(float) - float(known49["slot"].max())
    cols = [
        "d48_delta_prev", "d48_delta_next", "d48_delta_center", "d49_ratio_mean",
        "d49_ratio_std", "d49_ratio_trend", "d49_geo4_ratio_trend",
        "base_shape_gap", "alpha_x_gap", "slot_from_anchor"
    ]
    for col in cols:
        out[col] = out[col].replace([np.inf, -np.inf], np.nan).fillna(out[col].median())
    return out


def _loo_ridge_residual(valid, cols, alpha):
    result = np.zeros(len(valid), dtype=float)
    for slot in sorted(valid["slot"].unique()):
        train_mask = valid["slot"].ne(slot).to_numpy()
        val_mask = valid["slot"].eq(slot).to_numpy()
        scaler = StandardScaler()
        x_train = scaler.fit_transform(valid.loc[train_mask, cols])
        x_val = scaler.transform(valid.loc[val_mask, cols])
        residual = valid.loc[train_mask, "y"].to_numpy() - valid.loc[train_mask, "pred"].to_numpy()
        model = Ridge(alpha=alpha).fit(x_train, residual)
        result[val_mask] = model.predict(x_val)
    return result


def _full_ridge_residual(valid, test_df, cols, alpha, target_base_col="pred"):
    scaler = StandardScaler()
    x = scaler.fit_transform(valid[cols].replace([np.inf, -np.inf], np.nan).fillna(0))
    xt = scaler.transform(test_df[cols].replace([np.inf, -np.inf], np.nan).fillna(0))
    residual = valid["y"].to_numpy() - valid[target_base_col].to_numpy()
    return Ridge(alpha=alpha).fit(x, residual).predict(xt)


def build_93326_predictions(train_df, test_df, valid, clean_base_test):
    valid = add_slope_features_advanced(train_df, valid)
    test_stage = pd.DataFrame(
        {
            "Index": test_df["Index"],
            "slot": _slot_from_timestamp(test_df["timestamp"]),
            "RoadType": test_df["RoadType"].astype(str),
            "geohash": test_df["geohash"].astype(str),
            "base": clean_base_test,
            "shape": temporal_shape_prediction(train_df, test_df),
        }
    )
    test_stage = _add_prediction_bins(test_stage, "base")
    test_stage["alpha"] = [
        TEMPORAL_ALPHA_BY_GROUP.get((road, pred_bin), TEMPORAL_GLOBAL_ALPHA)
        for road, pred_bin in zip(test_stage["RoadType"], test_stage["pred_bin2"])
    ]
    test_stage["pred"] = clean_base_test
    test_stage = add_slope_features_advanced(train_df, test_stage)

    slope_only = ["d48_delta_prev", "d48_delta_next", "d48_delta_center"]
    slope_trend = [
        "d48_delta_prev", "d48_delta_next", "d48_delta_center", "d49_ratio_mean",
        "d49_ratio_std", "d49_ratio_trend", "d49_geo4_ratio_trend",
        "base_shape_gap", "alpha_x_gap", "slot_from_anchor"
    ]
    members = [
        ("slope_trend", slope_trend, 0.1, 0.30),
        ("slope_trend", slope_trend, 1.0, 0.30),
        ("slope_trend", slope_trend, 5.0, 0.30),
        ("slope_trend", slope_trend, 20.0, 0.30),
        ("slope_trend", slope_trend, 100.0, 0.30),
        ("slope_only", slope_only, 0.1, 0.30),
        ("slope_only", slope_only, 1.0, 0.30),
        ("slope_only", slope_only, 5.0, 0.30),
        ("slope_only", slope_only, 20.0, 0.30),
        ("slope_trend", slope_trend, 500.0, 0.30),
        ("slope_only", slope_only, 100.0, 0.30),
        ("slope_only", slope_only, 500.0, 0.30),
        ("slope_trend", slope_trend, 0.1, 0.20),
        ("slope_trend", slope_trend, 1.0, 0.20),
        ("slope_trend", slope_trend, 5.0, 0.20),
    ]
    valid_members, test_members = [], []
    for _, cols, alpha, shrink in members:
        rv = _loo_ridge_residual(valid, cols, alpha)
        rt = _full_ridge_residual(valid, test_stage, cols, alpha)
        valid_members.append(np.clip(valid["pred"].to_numpy() + shrink * rv, 0, 1))
        test_members.append(np.clip(clean_base_test + shrink * rt, 0, 1))
    pred_v = np.clip(0.50 * valid["pred"].to_numpy() + 0.50 * np.mean(valid_members, axis=0), 0, 1)
    pred_t = np.clip(0.50 * clean_base_test + 0.50 * np.mean(test_members, axis=0), 0, 1)
    return valid, test_stage, pred_v, pred_t


def add_circular_features(train_df, df):
    out = df.copy()
    work = add_basic_features(train_df.copy())
    day48 = work[work["day"].eq(48)].copy()
    pivot = day48.pivot_table(index="geohash", columns="slot", values="demand", aggfunc="mean")
    slot_mean = day48.groupby("slot")["demand"].mean()
    global_mean = float(day48["demand"].mean())

    def lookup(offset):
        vals = []
        for geo, slot in zip(out["geohash"].astype(str), out["slot"].astype(int)):
            shifted = (slot + offset) % 96
            value = pivot.at[geo, shifted] if geo in pivot.index and shifted in pivot.columns else np.nan
            vals.append(float(value) if pd.notna(value) else float(slot_mean.get(shifted, global_mean)))
        return np.asarray(vals)

    offsets = [-12, -8, -6, -4, -3, -2, -1, 0, 1, 2, 3, 4, 6, 8, 12]
    lag_cols = []
    for offset in offsets:
        col = f"circ_lag_{offset:+d}".replace("+", "p").replace("-", "m")
        out[col] = lookup(offset)
        lag_cols.append(col)
    out["circ_roll_3"] = out[["circ_lag_m1", "circ_lag_p0", "circ_lag_p1"]].mean(axis=1)
    out["circ_roll_5"] = out[["circ_lag_m2", "circ_lag_m1", "circ_lag_p0", "circ_lag_p1", "circ_lag_p2"]].mean(axis=1)
    out["circ_roll_9"] = out[["circ_lag_m4", "circ_lag_m3", "circ_lag_m2", "circ_lag_m1", "circ_lag_p0", "circ_lag_p1", "circ_lag_p2", "circ_lag_p3", "circ_lag_p4"]].mean(axis=1)
    out["circ_roll_wide"] = out[lag_cols].mean(axis=1)
    out["circ_slope_1"] = out["circ_lag_p1"] - out["circ_lag_m1"]
    out["circ_slope_2"] = 0.5 * (out["circ_lag_p2"] - out["circ_lag_m2"])
    out["circ_slope_4"] = 0.25 * (out["circ_lag_p4"] - out["circ_lag_m4"])
    out["circ_accel_1"] = out["circ_lag_p1"] - 2 * out["circ_lag_p0"] + out["circ_lag_m1"]
    out["circ_peak_ratio"] = out[lag_cols].max(axis=1) / (out[lag_cols].mean(axis=1) + 1e-4)
    out["circ_valley_ratio"] = out[lag_cols].min(axis=1) / (out[lag_cols].mean(axis=1) + 1e-4)
    return out


def add_phase_features(train_df, df):
    out = df.copy()
    work = add_basic_features(train_df.copy())
    day48 = work[work["day"].eq(48)]
    known49 = work[work["day"].eq(49)].copy()
    pivot48 = day48.pivot_table(index="geohash", columns="slot", values="demand", aggfunc="mean")
    shifts = [-2, -1, 0, 1, 2]
    best_shift, best_corr = {}, {}
    for geo, group in known49.groupby("geohash"):
        ordered = group.sort_values("slot")
        y = ordered["demand"].to_numpy(float)
        scores = []
        for shift in shifts:
            x = np.array(
                [
                    float(pivot48.at[geo, (slot + shift) % 96])
                    if geo in pivot48.index and (slot + shift) % 96 in pivot48.columns
                    else np.nan
                    for slot in ordered["slot"].astype(int)
                ]
            )
            ok = np.isfinite(x)
            corr = np.corrcoef(x[ok], y[ok])[0, 1] if ok.sum() >= 3 and np.std(x[ok]) > 1e-9 and np.std(y[ok]) > 1e-9 else -1
            scores.append(corr)
        choice = int(np.nanargmax(scores))
        best_shift[geo] = shifts[choice]
        best_corr[geo] = scores[choice]
    out["phase_shift_geo"] = out["geohash"].map(best_shift).fillna(0).astype(int)
    out["phase_corr_geo"] = out["geohash"].map(best_corr).fillna(0.0)
    values = []
    for geo, slot, shift in zip(out["geohash"].astype(str), out["slot"].astype(int), out["phase_shift_geo"]):
        shifted = (slot + shift) % 96
        values.append(float(pivot48.at[geo, shifted]) if geo in pivot48.index and shifted in pivot48.columns else np.nan)
    out["phase_shifted_d48"] = pd.Series(values).fillna(out["circ_lag_p0"].mean()).to_numpy()
    out["phase_delta_from_same"] = out["phase_shifted_d48"] - out["circ_lag_p0"]
    return out


def add_extended_phase_features(train_df, df):
    out = df.copy()
    work = add_basic_features(train_df.copy())
    day48 = work[work["day"].eq(48)].copy()
    known49 = work[work["day"].eq(49)].copy()
    day48["geo_4"] = day48["geohash"].astype(str).str[:4]
    known49["geo_4"] = known49["geohash"].astype(str).str[:4]
    out["geo_4"] = out["geohash"].astype(str).str[:4]
    pivot48 = day48.pivot_table(index="geohash", columns="slot", values="demand", aggfunc="mean")
    slot_mean = day48.groupby("slot")["demand"].mean()
    global_mean = float(day48["demand"].mean())
    shifts = [-3, -2, -1, 0, 1, 2, 3]

    def best_for_groups(group_col):
        result = {}
        for key, group in known49.groupby(group_col):
            y = group["demand"].to_numpy(float)
            scores = []
            for shift in shifts:
                vals = []
                for _, row in group.iterrows():
                    geo, slot = str(row["geohash"]), int(row["slot"])
                    shifted = (slot + shift) % 96
                    vals.append(float(pivot48.at[geo, shifted]) if geo in pivot48.index and shifted in pivot48.columns else np.nan)
                x = np.asarray(vals)
                ok = np.isfinite(x)
                corr = np.corrcoef(x[ok], y[ok])[0, 1] if ok.sum() >= 3 and np.std(x[ok]) > 1e-9 and np.std(y[ok]) > 1e-9 else -1
                scores.append(corr)
            result[key] = shifts[int(np.nanargmax(scores))]
        return result

    geo_shift = best_for_groups("geohash")
    geo4_shift = best_for_groups("geo_4")
    out["geo_best_shift_v2"] = out["geohash"].map(geo_shift).fillna(out["phase_shift_geo"]).astype(int)
    out["geo4_best_shift"] = out["geo_4"].map(geo4_shift).fillna(0).astype(int)
    for name, shift_col in [("geo_v2", "geo_best_shift_v2"), ("geo4", "geo4_best_shift")]:
        vals = []
        for geo, slot, shift in zip(out["geohash"].astype(str), out["slot"].astype(int), out[shift_col]):
            shifted = (slot + shift) % 96
            value = pivot48.at[geo, shifted] if geo in pivot48.index and shifted in pivot48.columns else np.nan
            vals.append(float(value) if pd.notna(value) else float(slot_mean.get(shifted, global_mean)))
        out[f"phase_{name}_shifted_d48"] = vals
        out[f"phase_{name}_delta_same"] = out[f"phase_{name}_shifted_d48"] - out["circ_lag_p0"]
        out[f"phase_{name}_ratio_same"] = out[f"phase_{name}_shifted_d48"] / (out["circ_lag_p0"] + 1e-4)
    return out


def ridge_adjust_stage(valid, test_df, base_v, base_t, cols, alpha, shrink):
    scaler = StandardScaler()
    x = scaler.fit_transform(valid[cols].replace([np.inf, -np.inf], np.nan).fillna(0))
    xt = scaler.transform(test_df[cols].replace([np.inf, -np.inf], np.nan).fillna(0))
    residual = valid["y"].to_numpy() - base_v
    model = Ridge(alpha=alpha).fit(x, residual)
    return (
        np.clip(base_v + shrink * model.predict(x), 0, 1),
        np.clip(base_t + shrink * model.predict(xt), 0, 1),
    )


print("Building rolling official-validation predictions for advanced temporal residual stages...")
rolling_predictions = build_rolling_validation_predictions(train)
advanced_valid = build_advanced_validation(train, rolling_predictions)
advanced_valid = advanced_valid.merge(
    add_basic_features(train)[["Index", "geohash"]],
    on="Index",
    how="left",
)
advanced_valid, advanced_test, base_93326_valid, base_93326_test = build_93326_predictions(
    train, test, advanced_valid, final_pred
)

advanced_valid = add_circular_features(train, advanced_valid)
advanced_test = add_circular_features(train, advanced_test)
advanced_valid = add_phase_features(train, advanced_valid)
advanced_test = add_phase_features(train, advanced_test)
advanced_valid = add_extended_phase_features(train, advanced_valid)
advanced_test = add_extended_phase_features(train, advanced_test)

circular_phase_cols = sorted(
    c
    for c in advanced_valid.columns
    if c.startswith(("circ_roll", "circ_slope", "circ_accel", "circ_peak", "circ_valley", "phase_"))
)
base_93395_valid, base_93395_test = ridge_adjust_stage(
    advanced_valid,
    advanced_test,
    base_93326_valid,
    base_93326_test,
    circular_phase_cols,
    alpha=1000.0,
    shrink=0.3500,
)

phase_local_cols = sorted(c for c in advanced_valid.columns if c.startswith("phase_"))
final_93401_valid, final_93401_test = ridge_adjust_stage(
    advanced_valid,
    advanced_test,
    base_93395_valid,
    base_93395_test,
    phase_local_cols,
    alpha=3000.0,
    shrink=0.2000,
)

submission = pd.DataFrame({"Index": test["Index"], "demand": final_93401_test})
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Overwrote {OUTPUT_PATH.resolve()} with the advanced circular + phase-local submission.")
print(submission["demand"].describe())
submission.head()
